# Interactive 3: Functional connectivity

Same subject, same run. Now we ignore the task and ask: *which parts of the brain rise and fall together?*

In [ ]:
#@title Setup: install packages and download the data (run this first, takes about a minute)
import os, sys
SUBJECT = 'cbp006'      # <-- change: 'cbp001', 'cbp006' (chronic back pain) or 'healthy007'

DRIVE_FOLDERS = {                    # one shared Google Drive folder per subject (each holds one 10-minute run)
    'cbp001': '10x0i_27x8cml7rVoQ9JNFrIxMm8T9gdc', 'cbp002': '1JP4tO-cg3CDREIIjDolyy1U0qhOKnJJ_', 'cbp003': '1IXBkI7P7yZ50Z38c5Kfg54uApIv12F5y',
    'cbp006': '19-I4KodmUlsUFrrnQmaK7C5UPBM_d-d4', 'cbp014': '1l48f17kf0nzFu5mH6MJZ46eqajJP_3UJ',
    'healthy001': '1BLuc3TPFzP6sziY9-c6npeYv5n7ct48g', 'healthy003': '1u5qB-WXYzofhmn807ypm8DXZyEuIHxB3', 'healthy004': '18W0jyvHSa2-h1JkrbOvXlahF5fQPZA_c',
    'healthy007': '1WI1QMf0vj2163cKWasGQLiSDf-x4wl7m', 'healthy012': '1zqO35e9Cxi0gddl7n6IEPIrI1qQVGaDG'}

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    get_ipython().system('pip install -q --no-deps nilearn ipyniivue anywidget psygnal')   # only what Colab lacks; --no-deps keeps Colab's pandas/requests
    from google.colab import output; output.enable_custom_widget_manager()               # lets the clickable brain viewer render in Colab
    DATA = f'data/{SUBJECT}'
    if not os.path.exists(DATA):
        get_ipython().system(f'gdown --folder https://drive.google.com/drive/folders/{DRIVE_FOLDERS[SUBJECT]} -O {DATA} -q')
else:
    DATA = os.path.join(os.environ.get('FMRI_DATA', 'drive_upload'), SUBJECT)

import numpy as np, nibabel as nib, matplotlib.pyplot as plt, pandas as pd, warnings
from nilearn import plotting, image, masking
from nilearn.datasets import load_mni152_template
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 90
TR = 2.5                                   # seconds between volumes
t = np.arange(240) * TR                    # time axis in seconds
print('files:', sorted(os.listdir(DATA)))

## 1. Prepare the data: band-pass filter, remove head motion, z-score every voxel

In [ ]:
from nilearn.maskers import NiftiMasker, NiftiSpheresMasker

bold = nib.load(f'{DATA}/bold_mni.nii.gz')
motion = np.loadtxt(f'{DATA}/motion_params.txt')
stim = np.loadtxt(f'{DATA}/stimulus.txt')

HIGH_PASS, LOW_PASS = 0.009, 0.08      # Hz  <-- change
SMOOTHING = 6                          # mm  <-- change
REMOVE_GLOBAL_SIGNAL = True            # <-- change: also regress out the average signal of the whole brain

brain_mask = masking.compute_epi_mask(bold)
global_signal = masking.apply_mask(bold, brain_mask).mean(1)
confounds = np.column_stack([motion, global_signal]) if REMOVE_GLOBAL_SIGNAL else motion
clean = image.clean_img(image.smooth_img(bold, SMOOTHING), t_r=TR, confounds=confounds, high_pass=HIGH_PASS, low_pass=LOW_PASS, detrend=True, standardize='zscore_sample', mask_img=brain_mask)
masker = NiftiMasker(mask_img=brain_mask).fit()
X = masker.transform(clean)            # matrix: 240 time points x all brain voxels
print('time points x voxels:', X.shape)

## 2. Seed-based connectivity: correlate one region with every voxel

In [ ]:
SEED = (16, 10, -8)      # MNI  <-- change: (16, 10, -8) right accumbens; (0, 52, -14) medial prefrontal; (40, 8, -2) right insula; (-38, -22, 56) left motor cortex
RADIUS = 5               # mm
R_THRESHOLD = 0.4        # <-- change

seed = NiftiSpheresMasker([SEED], radius=RADIUS).fit_transform(clean)[:, 0]
seed = (seed - seed.mean()) / seed.std(ddof=1)
r = X.T @ seed / (len(seed) - 1)                     # correlation of the seed with every voxel
r_img = masker.inverse_transform(r)

plt.figure(figsize=(13, 3)); plt.plot(t, seed, color='k'); plt.title(f'seed time series at MNI {SEED}'); plt.xlabel('time (s)'); plt.show()
plotting.plot_stat_map(r_img, threshold=R_THRESHOLD, cut_coords=SEED, title=f'correlation with seed {SEED}', vmax=1)
plotting.plot_glass_brain(r_img, threshold=R_THRESHOLD, colorbar=True, plot_abs=False, display_mode='lyrz', vmax=1)
plotting.show()

In [ ]:
# What a correlation of 0.7, 0.0 and -0.4 look like: the seed (black) against one voxel (colored) at each strength
TARGET_R = [0.7, 0.0, -0.4]                            # <-- change
target_colors = ['seagreen', 'orange', 'royalblue']

voxel_ijk = np.argwhere(brain_mask.get_fdata() > 0)   # voxel indices, same order as the columns of X
targets = []
for target_r, color in zip(TARGET_R, target_colors):
    j = np.abs(r - target_r).argmin()
    xyz = tuple(np.round(image.coord_transform(*voxel_ijk[j], brain_mask.affine)).astype(int))
    targets.append((j, xyz, color))

d = plotting.plot_glass_brain(None, display_mode='lyrz', title='seed (black) and the three example voxels')
d.add_markers([SEED], marker_color='k', marker_size=120)
for j, xyz, color in targets:
    d.add_markers([xyz], marker_color=color, marker_size=120)
plotting.show()

for j, xyz, color in targets:
    plt.figure(figsize=(13, 2.5)); plt.plot(t, seed, color='k', label='seed'); plt.plot(t, X[:, j], color=color, label=f'voxel at MNI {xyz}')
    plt.title(f'r = {r[j]:.2f}'); plt.legend(loc='upper right'); plt.show()

## 3. A correlation matrix between regions

In [ ]:
ROIS = {                       # <-- change: add or remove regions (name: MNI coordinate)
    'NAc R': (16, 10, -8),   'NAc L': (-16, 10, -8),
    'insula R': (40, 8, -2), 'insula L': (-40, 8, -2),
    'S2 R': (58, -22, 18),   'S2 L': (-58, -22, 18),
    'thalamus R': (10, -18, 6), 'thalamus L': (-10, -18, 6),
    'ACC': (2, 20, 32),      'mPFC': (0, 52, -14),
    'PCC': (0, -52, 26),     'motor L': (-38, -22, 56),
}
roi_ts = NiftiSpheresMasker(list(ROIS.values()), radius=RADIUS).fit_transform(clean)
corr = np.corrcoef(roi_ts.T)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(ROIS))); ax.set_xticklabels(ROIS, rotation=90); ax.set_yticks(range(len(ROIS))); ax.set_yticklabels(ROIS)
plt.colorbar(im, label='correlation'); plt.show()

plotting.plot_connectome(corr, list(ROIS.values()), edge_threshold=0.4, node_size=60, title='edges with |r| > 0.4')
plotting.show()

## 4. Degree: how many strong connections does each voxel have?

In [ ]:
DEGREE_THRESHOLD = 0.7     # <-- change: r above which we call two voxels "connected"

n = X.shape[0]
degree = np.zeros(X.shape[1])
for start in range(0, X.shape[1], 2000):              # correlate every voxel with every other voxel, in chunks
    chunk = X[:, start:start + 2000].T @ X / (n - 1)
    degree[start:start + 2000] = (chunk >= DEGREE_THRESHOLD).sum(1) - 1

plotting.plot_stat_map(masker.inverse_transform(degree), threshold=np.percentile(degree, 80), cmap='hot', title=f'degree (number of voxels with r >= {DEGREE_THRESHOLD})', display_mode='z', cut_coords=6)
plotting.show()

## 5. Why preprocessing matters for connectivity

In [ ]:
SEED = (0, 52, -14)      # <-- change

def seed_map(img, label):
    m = NiftiMasker(mask_img=brain_mask, standardize='zscore_sample').fit()
    Xi = m.transform(img)
    s = NiftiSpheresMasker([SEED], radius=RADIUS, standardize='zscore_sample').fit_transform(img)[:, 0]
    plotting.plot_stat_map(m.inverse_transform(Xi.T @ s / (len(s) - 1)), threshold=R_THRESHOLD, cut_coords=SEED, title=label, vmax=1)

seed_map(bold, 'raw data, no filtering, no motion regression')
seed_map(image.clean_img(bold, t_r=TR, high_pass=HIGH_PASS, low_pass=LOW_PASS, detrend=True, mask_img=brain_mask), 'band-pass filtered only')
seed_map(image.clean_img(bold, t_r=TR, confounds=motion, high_pass=HIGH_PASS, low_pass=LOW_PASS, detrend=True, mask_img=brain_mask), 'filtered + motion regressed')
seed_map(clean, 'filtered + motion regressed' + (' + global signal regressed' if REMOVE_GLOBAL_SIGNAL else '') + ' + smoothed')
plotting.show()